# YouTube Comment Scraping for IHSG

## Notebook: Data Scraping and Filtering

This notebook collects top-level comments from YouTube videos about the IHSG, checks the raw data, and filters comments related to the stock market.

## 1. Import Libraries

These libraries load the configuration, fetch comments through the YouTube Data API, work with tables, and display the results.

In [14]:
import os
import re
import time
from pathlib import Path

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from IPython.display import display

## 2. Load the YouTube API Key

The API key can be set in the `YOUTUBE_API_KEY` environment variable or stored in a `.env` file at the project root. The project path is resolved from the notebook's working directory so output paths stay consistent.

In [15]:
# Resolve the project root from the root or notebooks directory.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run the notebook from the project root or the notebooks directory.")

# Load YOUTUBE_API_KEY from .env when it is not set in the environment.
env_file = PROJECT_ROOT / ".env"
if not os.getenv("YOUTUBE_API_KEY") and env_file.is_file():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        key, separator, value = line.strip().partition("=")
        if separator and key == "YOUTUBE_API_KEY":
            os.environ["YOUTUBE_API_KEY"] = value.strip().strip("\"'")
            break

## 3. Validate the API Key

Check that the API key is available before creating the client. The key itself is never printed in notebook output.

In [16]:
API_KEY = os.getenv("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY is missing. Set it in the environment or the .env file.")

print("YouTube API key loaded successfully.")

YouTube API key loaded successfully.


## 4. Video List and Configuration

The listed videos are the comment sources. Up to 2,000 top-level comments are requested from each video. This section also defines the paths for the raw and filtered datasets.

In [17]:
# Videos whose comments will be collected.
VIDEO_LIST = [
    "https://youtu.be/oL8nCaXXJPE?si=NFrPfKvdYIEldSrm",
    "https://youtube.com/shorts/kuCxBDAUZz4?si=HRv7C2bGES2tnDrS",
    "https://youtube.com/shorts/svYRuI09D9U?si=8xP_Hi6i5fx9zX94",
    "https://youtu.be/I1NBxuJ0hM0?si=XgE8aZoYx3Nxq8BJ",
    "https://youtu.be/hsZzNqL2yVA?si=X0Q2t2kFC8_JwcHT",
    "https://youtu.be/9gVzgIJ5zjg?si=wg6x8YLtq1wmSdEL",
    "https://youtube.com/shorts/2LZLR-Hnun0?si=mI4sotlC8uC3-m_i",
    "https://youtu.be/tf1npHikQtY?si=9vRDALg9bcnyEwJK"
    
]

# Scraping settings and output files.
MAX_COMMENTS_PER_VIDEO = 2000  # Maximum top-level comments per video.
ORDER = "relevance"             # Options: "relevance" or "time".
DATA_DIR = PROJECT_ROOT / "datasets"
RAW_CSV = DATA_DIR / "comments_raw.csv"
FILTERED_CSV = DATA_DIR / "comments_filtered.csv"

# Preprocessing and model settings.
USE_STEMMING = True  # Sastrawi stemming can be used during preprocessing.
SEED = 42
N_FOLDS = 5
MAX_WORDS = 20000
MAX_LEN = 60
EPOCHS = 15
BATCH_SIZE = 32

## 5. Initialize the YouTube API Client

Create the API client with the validated key to request public comments from each video.

In [18]:
youtube = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)
print("YouTube API client is ready.")

YouTube API client is ready.


## 6. Scrape Comments

Each URL is converted to a video ID, then `commentThreads.list` retrieves top-level comments. The API returns at most 100 comments per page, and `nextPageToken` is followed until the video limit or the final page is reached. Replies are not requested. If a request fails, its status is recorded and comments already collected are retained.

In [19]:
def extract_video_id(value):
    pattern = r"(?:youtu\.be/|youtube\.com/(?:shorts/|embed/)|[?&]v=)([A-Za-z0-9_-]{11})"
    match = re.search(pattern, value)
    video_id = match.group(1) if match else value.strip()
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise ValueError(f"Invalid video URL or ID: {value}")
    return video_id


# Validate the list so each video is processed only once.
video_ids = [extract_video_id(url) for url in VIDEO_LIST]
if len(video_ids) != len(set(video_ids)):
    raise ValueError("VIDEO_LIST contains a duplicate video.")

all_rows = []
video_results = []
for index, video_id in enumerate(video_ids, start=1):
    start_count = len(all_rows)
    next_page_token = None
    status = "completed"
    print(f"[{index}/{len(video_ids)}] Fetching comments for video {video_id}...")

    # Fetch top-level comments page by page up to the video limit.
    while len(all_rows) - start_count < MAX_COMMENTS_PER_VIDEO:
        try:
            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, MAX_COMMENTS_PER_VIDEO - (len(all_rows) - start_count)),
                pageToken=next_page_token,
                order=ORDER,
                textFormat="plainText",
            ).execute(num_retries=3)
        except HttpError as error:
            status = f"failed (HTTP {error.resp.status})"
            break

        # Keep the original text and metadata for each top-level comment.
        for item in response.get("items", []):
            comment = item["snippet"]["topLevelComment"]
            snippet = comment["snippet"]
            all_rows.append({
                "video_id": video_id,
                "comment_id": comment["id"],
                "text": snippet["textDisplay"],
                "author": snippet.get("authorDisplayName", ""),
                "like_count": snippet.get("likeCount", 0),
                "published_at": snippet["publishedAt"],
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token or not response.get("items"):
            break
        time.sleep(0.2)

    count = len(all_rows) - start_count
    video_results.append({"video_id": video_id, "comment_count": count, "status": status})
    print(f"{count:,} top-level comments collected ({status}).")

# Build the raw table without changing or filtering comments.
df_raw = pd.DataFrame(
    all_rows,
    columns=["video_id", "comment_id", "text", "author", "like_count", "published_at"],
)
df_video = pd.DataFrame(video_results)

[1/8] Fetching comments for video oL8nCaXXJPE...
913 top-level comments collected (completed).
[2/8] Fetching comments for video kuCxBDAUZz4...
1,043 top-level comments collected (completed).
[3/8] Fetching comments for video svYRuI09D9U...
933 top-level comments collected (completed).
[4/8] Fetching comments for video I1NBxuJ0hM0...
918 top-level comments collected (completed).
[5/8] Fetching comments for video hsZzNqL2yVA...
916 top-level comments collected (completed).
[6/8] Fetching comments for video 9gVzgIJ5zjg...
888 top-level comments collected (completed).
[7/8] Fetching comments for video 2LZLR-Hnun0...
883 top-level comments collected (completed).
[8/8] Fetching comments for video tf1npHikQtY...
516 top-level comments collected (completed).


## 7. Inspect the Scraping Results

Show the comment count and request status for each video. A few raw comments are printed in full as a numbered list so line breaks remain readable.

In [20]:
print("Comments collected from each video:")
display(df_video)
print(f"Total raw comments: {len(df_raw):,}")
print(f"Duplicate comment IDs: {df_raw['comment_id'].duplicated().sum():,}")

print("Raw comment examples:")
for number, row in enumerate(df_raw.head(5).itertuples(index=False), start=1):
    print(f"{number}. [{row.video_id}] {row.text}\n")

Comments collected from each video:


,video_id,comment_count,status
0,oL8nCaXXJPE,913,completed
1,kuCxBDAUZz4,1043,completed
2,svYRuI09D9U,933,completed
3,I1NBxuJ0hM0,918,completed
4,hsZzNqL2yVA,916,completed
5,9gVzgIJ5zjg,888,completed
6,2LZLR-Hnun0,883,completed
7,tf1npHikQtY,516,completed


Total raw comments: 7,010
Duplicate comment IDs: 0
Raw comment examples:
1. [oL8nCaXXJPE] msci cuma minta nasi goreng. indonesia ngasihnya soto. Msci protes minta nasi goreng. Ttp indonesia ngasihnya sate, martabak, pulsa. Gitu2 terus bertahun2 gk dikasih2.

Ritelnya jg lebih lucu, nyalahin mbg. msci jelas2 maunya nasi goreng, gk peduli mbg.

2. [oL8nCaXXJPE] Selamat ye bennix Indonesia dah jadi maju mengikuti zimbabwe.. kami di ASEAN dah beda level  dengan Indonesia.. betul ramalan benix

3. [oL8nCaXXJPE] Prabowo : indonesia harus bisa berdiri dengan kaki kita sendiri 

MSCI : Oke, gua masukin standalone yah

4. [oL8nCaXXJPE] Watch from malaysia, hi apa khabar

5. [oL8nCaXXJPE] Sampaikan kepada pak Prabowo, jumlah korban karhutla meningkat  kebanyakan korban bayi dan anak-anak .



## 8. Save the Raw Data

All API results are saved to `datasets/comments_raw.csv` without removing comments. Enter is stored as a literal `\n` and Tab is replaced with a space so each comment occupies one CSV line.

In [21]:
if df_raw.empty:
    raise ValueError("No comments were collected; the raw file was not created.")

# Escape line breaks and tabs so raw CSV comments stay on one line.
def escape_newlines(text):
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\t", " ")
    return re.sub(r"\n+", r"\\n", text).strip()

df_raw["text"] = df_raw["text"].map(escape_newlines)
DATA_DIR.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(RAW_CSV, index=False, encoding="utf-8-sig")
print(f"Raw data saved: {RAW_CSV} ({len(df_raw):,} rows)")

Raw data saved: c:\Machine Learning\ML\datasets\comments_raw.csv (7,010 rows)


## 9. Load and Check the Raw Data

Read the saved CSV before filtering. The checks cover row count, empty IDs or text, duplicate IDs or text, and comments containing escaped line breaks.

In [22]:
df_raw = pd.read_csv(RAW_CSV, encoding="utf-8-sig", keep_default_na=False)
text_key = df_raw["text"].astype(str).str.replace("\\n", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()

# Summarize data quality before filtering.
raw_profile = pd.DataFrame([{
    "row_count": len(df_raw),
    "empty_id": df_raw["comment_id"].astype(str).str.strip().eq("").sum(),
    "empty_text": df_raw["text"].astype(str).str.strip().eq("").sum(),
    "duplicate_id": df_raw["comment_id"].duplicated().sum(),
    "duplicate_text": text_key.duplicated().sum(),
    "escaped_line_break": df_raw["text"].str.contains("\\n", regex=False).sum(),
}])
display(raw_profile)

,row_count,empty_id,empty_text,duplicate_id,duplicate_text,escaped_line_break
0,7010,0,0,0,68,858


## 10. Choose Filtering Keywords

The keywords were selected by counting their occurrences across the raw comments and reviewing examples. Frequent but broad words, such as people's names or country names, are not used alone. Less frequent stock-market terms and observed spelling variants are included when they are relevant. The keywords remain in Indonesian because they must match the original comments.

In [23]:
# Stock-market terms and observed spelling variants.
TOPIC_TERMS = [
    "ihsg", "ihsgnya", "ishg", "msci", "saham", "saham2", "sahamnya",
    "sahamku", "bursa", "bursanya", "pasar saham", "pasar modal",
    "emiten", "emiten2", "investor", "investor2", "investasi", "indeks",
    "indeksnya", "idx", "bei", "ojk", "market", "sekuritas", "trading",
    "trader", "ipo", "dividen", "portofolio", "broker", "obligasi",
    "reksa dana", "free float", "ara", "arb", "stock", "gorengan",
    "nyerok", "cut loss",

    # Economic and market-status terms found in the comments.
    "ekonomi", "ekonominya", "perekonomian", "keuangan", "krisis",
    "krismon", "rupiah", "dolar", "dollar", "inflasi", "moneter",
    "likuiditas", "modal asing", "standalone", "stand alone",
    "standalonnee", "frontier", "emerging", "zimbabwe", "zimbabwekan",
    "zimbabwae", "argentina", "downgrade", "turun kelas", "turun kasta",
    "beda level",
]


def contains_topic(text):
    words = re.findall("[a-z0-9]+", str(text).replace("\\n", " ").casefold())
    normalized = " " + " ".join(words) + " "
    return any(" " + term + " " in normalized for term in TOPIC_TERMS)

## 11. Filter Comments

First remove rows with an empty ID or text and duplicate comments. Then keep only comments containing at least one selected keyword. Escaped line breaks and repeated spaces are collapsed in the filtered text.

In [24]:
# Remove empty rows and duplicate comments from a copy of the raw data.
valid_rows = df_raw.loc[
    df_raw["comment_id"].astype(str).str.strip().ne("")
    & df_raw["text"].astype(str).str.strip().ne("")
].copy()
unique_ids = valid_rows.drop_duplicates(subset="comment_id", keep="first").copy()
unique_ids["_text_key"] = (
    unique_ids["text"].astype(str).str.replace("\\n", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()
)
unique_texts = unique_ids.drop_duplicates(subset="_text_key", keep="first").copy()

# Keep only comments matching a selected keyword.
topic_mask = unique_texts["text"].map(contains_topic)
df_filtered = unique_texts.loc[topic_mask].drop(columns="_text_key").reset_index(drop=True)

# Keep each filtered comment on one line without changing the raw dataset.
df_filtered["text"] = df_filtered["text"].str.replace("\\n", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip()
df_excluded = unique_texts.loc[~topic_mask].drop(columns="_text_key").reset_index(drop=True)

filter_report = pd.DataFrame([
    {"stage": "Raw data", "count": len(df_raw)},
    {"stage": "Nonempty ID and text", "count": len(valid_rows)},
    {"stage": "Unique comment IDs", "count": len(unique_ids)},
    {"stage": "Unique comment text", "count": len(unique_texts)},
    {"stage": "Matches a keyword", "count": len(df_filtered)},
    {"stage": "No keyword match", "count": len(df_excluded)},
])

## 12. Inspect the Filtered Results

Show the row count at each step and the filtered count per video. Full examples of included comments and a random sample of excluded comments are printed as numbered lists. Whether there are enough examples for each emotion class can only be assessed after labeling.

In [25]:
print("Comment count at each filtering step:")
display(filter_report)

print("Filtered comments per video:")
display(df_filtered.groupby("video_id").size().reset_index(name="comment_count"))

print("Examples of included comments:")
for number, row in enumerate(df_filtered.head(10).itertuples(index=False), start=1):
    print(f"{number}. [{row.video_id}] {row.text}\n")

print("Random examples of comments with no keyword match:")
sample_excluded = df_excluded.sample(min(10, len(df_excluded)), random_state=SEED)
for number, row in enumerate(sample_excluded.itertuples(index=False), start=1):
    print(f"{number}. [{row.video_id}] {row.text}\n")

Comment count at each filtering step:


,stage,count
0,Raw data,7010
1,Nonempty ID and text,7010
2,Unique comment IDs,7010
3,Unique comment text,6942
4,Matches a keyword,2088
5,No keyword match,4854


Filtered comments per video:


,video_id,comment_count
0,2LZLR-Hnun0,287
1,9gVzgIJ5zjg,150
2,I1NBxuJ0hM0,274
3,hsZzNqL2yVA,324
4,kuCxBDAUZz4,228
5,oL8nCaXXJPE,397
6,svYRuI09D9U,304
7,tf1npHikQtY,124


Examples of included comments:
1. [oL8nCaXXJPE] msci cuma minta nasi goreng. indonesia ngasihnya soto. Msci protes minta nasi goreng. Ttp indonesia ngasihnya sate, martabak, pulsa. Gitu2 terus bertahun2 gk dikasih2. Ritelnya jg lebih lucu, nyalahin mbg. msci jelas2 maunya nasi goreng, gk peduli mbg.

2. [oL8nCaXXJPE] Selamat ye bennix Indonesia dah jadi maju mengikuti zimbabwe.. kami di ASEAN dah beda level dengan Indonesia.. betul ramalan benix

3. [oL8nCaXXJPE] Prabowo : indonesia harus bisa berdiri dengan kaki kita sendiri MSCI : Oke, gua masukin standalone yah

4. [oL8nCaXXJPE] Ini kesempatan buat bersihin bursa kita dari mafia2 penjual nasi goreng...😂

5. [oL8nCaXXJPE] bentuk ketidakpercayaan internasional terhadap transparansi, regulasi, dan tata kelola di bawah pemerintahan saat ini. jatuhnya IHSG

6. [oL8nCaXXJPE] Saran Investor: Kreator menyarankan investor ritel untuk tetap tenang, melakukan evaluasi portofolio dengan teliti, melakukan diversifikasi, serta menghindari penggun

## 13. Save the Filtered Data

Comments matching the selected keywords are saved to `datasets/comments_filtered.csv` with the same columns as the raw dataset. Raw comments retain escaped Enter markers, while filtered comments use spaces in their place.

In [26]:
if df_filtered.empty:
    raise ValueError("No comments matched the filtering criteria.")

# Save the filtered dataset without temporary columns.
df_filtered.to_csv(FILTERED_CSV, index=False, encoding="utf-8-sig")
print(f"Filtered data saved: {FILTERED_CSV} ({len(df_filtered):,} rows)")

Filtered data saved: c:\Machine Learning\ML\datasets\comments_filtered.csv (2,088 rows)
